# Tutorial: Module 02 Bias Variance Simulation

Audience:
- Learners beginning statistical learning.

Prerequisites:
- Basic regression intuition.
- Python arrays and plotting fundamentals.

Learning goals:
- Simulate noisy data from a simple target function.
- Compare a low-degree and high-degree polynomial regressor.
- Interpret train and test error as a bias-variance tradeoff signal.


## Outline

1. Create a synthetic regression dataset.
2. Fit two polynomial models.
3. Measure train and test mean squared error.
4. Render a compact comparison plot.


In [ ]:
from __future__ import annotations

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

rng = np.random.default_rng(12)


## Step 1 - Synthetic data

We generate observations from a sinusoid with Gaussian noise. The small training set lets a high-degree polynomial overfit enough to make the tradeoff visible.


In [ ]:
def target(x: np.ndarray) -> np.ndarray:
    return np.sin(2 * np.pi * x)


x_train = np.linspace(0.0, 1.0, 12).reshape(-1, 1)
y_train = target(x_train[:, 0]) + rng.normal(0.0, 0.12, size=x_train.shape[0])
x_test = np.linspace(0.0, 1.0, 200).reshape(-1, 1)
y_test = target(x_test[:, 0])

x_train[:3], y_train[:3]


## Step 2 - Fit two models

Degree 2 is intentionally simple, while degree 10 is flexible enough to chase the noise in such a small sample.


In [ ]:
def fit_model(degree: int) -> Pipeline:
    return Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("regressor", LinearRegression()),
    ]).fit(x_train, y_train)


models = {degree: fit_model(degree) for degree in (2, 10)}
metrics = {}

for degree, model in models.items():
    train_pred = model.predict(x_train)
    test_pred = model.predict(x_test)
    metrics[degree] = {
        "train_mse": mean_squared_error(y_train, train_pred),
        "test_mse": mean_squared_error(y_test, test_pred),
    }

metrics


## Step 3 - Plot the fitted curves

The plot is saved and closed immediately so the notebook stays headless and CI-friendly.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(x_train[:, 0], y_train, color="black", label="train data")
ax.plot(x_test[:, 0], y_test, linestyle="--", color="gray", label="target")

for degree, model in models.items():
    ax.plot(x_test[:, 0], model.predict(x_test), label=f"degree {degree}")

ax.set_title("Bias-variance comparison")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
fig.tight_layout()
plt.close(fig)

{"degree_2_test_mse": metrics[2]["test_mse"], "degree_10_test_mse": metrics[10]["test_mse"]}
